## Example 1 — the version, and typed arguments are parsed and checked

In [ ]:
import argparse, importlib.metadata
from jsonargparse import ArgumentParser

import jsonargparse
assert jsonargparse.__version__ == importlib.metadata.version("jsonargparse")
parser = ArgumentParser(exit_on_error=False)
parser.add_argument("--lr", type=float, default=0.1)
parser.add_argument("--layers", type=list[int], default=[1])
cfg = parser.parse_args(["--lr", "0.01", "--layers", "[2, 3]"])
assert cfg.lr == 0.01 and cfg.layers == [2, 3], cfg
try:
    parser.parse_args(["--lr", "fast"])
    raise AssertionError("a non-number was accepted")
except argparse.ArgumentError as error:
    print("refused as expected:", type(error).__name__)
print("jsonargparse", jsonargparse.__version__, "parsed", cfg.as_dict())

## Example 2 — a YAML config file, overridden from the command line

In [ ]:
import os, tempfile
from jsonargparse import ActionConfigFile, ArgumentParser

parser = ArgumentParser(exit_on_error=False)
parser.add_argument("--config", action=ActionConfigFile)
parser.add_argument("--name", type=str, default="none")
parser.add_argument("--epochs", type=int, default=1)
with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "run.yaml")
    with open(path, "w") as f:
        f.write("name: demo\nepochs: 5\n")
    cfg = parser.parse_args(["--config", path, "--epochs", "7"])
assert cfg.name == "demo" and cfg.epochs == 7, cfg
dumped = parser.dump(cfg)
assert "epochs: 7" in dumped, dumped
print(dumped)

## Example 3 — a class's arguments, with help text read from its docstring (the `signatures` extra)

In [ ]:
from jsonargparse import ArgumentParser

class Model:
    def __init__(self, hidden: int = 8, dropout: float = 0.0):
        """A small model.

        Args:
            hidden: width of the hidden layer
            dropout: probability of dropping a unit
        """
        self.hidden, self.dropout = hidden, dropout

parser = ArgumentParser(exit_on_error=False)
parser.add_class_arguments(Model, "model")
cfg = parser.parse_args(["--model.hidden", "16"])
model = Model(**cfg.model.as_dict())
assert (model.hidden, model.dropout) == (16, 0.0)
help_text = parser.format_help()
assert "width of the hidden layer" in help_text, help_text
print("help text from the docstring found; model:", vars(model))

## Example 4 — `auto_cli` turns a function into a command line

In [ ]:
from jsonargparse import auto_cli

def train(epochs: int, lr: float = 0.1, tags: list[str] | None = None) -> str:
    """Train for some epochs."""
    return f"{epochs} epochs at {lr} {tags or []}"

result = auto_cli(train, args=["3", "--lr", "0.5", "--tags", "[a, b]"])
assert result == "3 epochs at 0.5 ['a', 'b']", result
print(result)

## Example 5 — a standard-library function written in C gets its signature from typeshed_client

In [ ]:
import random
from jsonargparse import ArgumentParser

# random.Random.randint is pure Python, random.Random.seed is not; jsonargparse resolves the
# parameters of C functions from typeshed's stub files through typeshed_client.
parser = ArgumentParser(exit_on_error=False)
added = parser.add_method_arguments(random.Random, "randint", "r")
cfg = parser.parse_args(["--r.a", "1", "--r.b", "6"])
assert (cfg.r.a, cfg.r.b) == (1, 6), cfg
from jsonargparse._stubs_resolver import get_stubs_resolver
resolver = get_stubs_resolver()
assert resolver is not None
print("randint arguments:", sorted(added), "; stubs resolver:", type(resolver).__name__)